In [1]:
import pandas as pd

In [3]:
pth1 = r"IBBI_ARCHIVE_20260427.xlsx"
pth2 = r"IBBI_ARCHIVE_20260428.xlsx"
df1 = pd.read_excel(pth1,sheet_name="supreme_court")
df2 = pd.read_excel(pth2,sheet_name="supreme_court")

In [16]:
f1 = df1.iloc[:,0:6]
f1.columns = ["court","sr","date","case","status","link"]
f1["date"] = pd.to_datetime(f1["date"], errors="coerce")
f1 = f1.sort_values(by="date", ascending=False)
f1 = f1.reset_index(drop=True)
f1.to_csv("20260427.csv")
# f1.head(10)


In [17]:
f2 =df2.iloc[:,0:6]
f2.columns = ["court","sr","date","case","status","link"]
f2["date"] = pd.to_datetime(f2["date"], errors="coerce")
f2 = f2.sort_values(by="date", ascending=False)
f2 = f2.reset_index(drop=True)
f2.to_csv("20260428.csv")
# f2.head(10)

In [31]:
import pandas as pd
import hashlib



def prepare_df(df):
    df = df.copy()

    df = df.iloc[:, :6]
    df.columns = ["court", "sr", "date", "case", "status", "link"]

    df["court"] = df["court"].astype(str).str.strip().str.lower()
    df["case"] = df["case"].astype(str).str.strip()
    df["status"] = df["status"].astype(str).str.strip().str.lower()
    df["link"] = df["link"].astype(str).str.strip()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df = df.drop_duplicates(subset=["link"])

    hash_src = (
        df["court"].fillna("") + "|" +
        df["date"].dt.strftime("%Y-%m-%d").fillna("") + "|" +
        df["case"].fillna("") + "|" +
        df["link"].fillna("")
    )

    df["hash_id"] = hash_src.map(
        lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
    )

    return df



def compare(df_old, df_new):

    prev_map = df_old.set_index("link")["hash_id"].to_dict()

    def classify(row):
        key = row["link"]
        if key not in prev_map:
            return "NEW"
        elif prev_map[key] != row["hash_id"]:
            return "UPDATED"
        return "UNCHANGED"

    df_new = df_new.copy()
    df_new["change_type"] = df_new.apply(classify, axis=1)

    final_df = df_new
    new_df = df_new[df_new["change_type"] == "NEW"]
    updated_df = df_new[df_new["change_type"] == "UPDATED"]
    unchanged_df = df_new[df_new["change_type"] == "UNCHANGED"]

    return new_df, updated_df, unchanged_df, final_df



old_file = "IBBI_ARCHIVE_20260427.xlsx"
new_file = "IBBI_ARCHIVE_20260428.xlsx"

TYPE_ = "supreme_court" #"other_courts" #"nclat" #"supreme_court"

df_old = pd.read_excel(old_file, sheet_name=TYPE_)
df_new = pd.read_excel(new_file, sheet_name=TYPE_)

if df_old.empty or df_new.empty:
    print("Skipped (empty)")
else:
    df_old = prepare_df(df_old)
    df_new = prepare_df(df_new)

    new_df, updated_df, unchanged_df,final_df = compare(df_old, df_new)

    print(f"New: {len(new_df)}")
    print(f"Updated: {len(updated_df)}")
    print(f"Unchanged: {len(unchanged_df)}")


final_df.to_csv(f"{TYPE_}.csv",index=False)


New: 2
Updated: 0
Unchanged: 198
